Purpose: Given a TraceML human trajectory, select a version $v_t$ as a breakpoint, provide the state of $v_t$ to the model, let it suggest the next step, and then compare it side-by-side with the expert's actual $v_t \to v_{t+1}$ transition.

**Data Dependencies (TraceML Release)**

| File | Purpose | Required |
|---|---|---|
| `data/<split>/action.parquet` | Labels for each version transition, `goal_nl`, `diff_summary`, score | Yes |
| `trajectories_human.tar.gz` unzipped directory | `.ipynb` source code for each version (state given to the model at breakpoint) | Yes (otherwise only label summaries can be used as state) |
| `data/<split>/state.parquet` | State labels for each version | No |
| `manifests/competitions.json` | Competition metadata (score direction) | No |

**State given to the model**: Snapshot changes for each step `v_1..v_t` (actual code diffs, or full snapshots), reasoning between each step (TraceML annotated `goal_nl`/`diff_summary` as a proxy for the expert's judgment at the time), public leaderboard scores and relative changes for each version (relative to the last scored version, relative to the current best score), and the full code of `v_t`. The expert's reasoning and results for the `v_t→v_{t+1}` step are not included in the prompt. To verify if injecting a piece of knowledge at the breakpoint changes the model's choice, use the `HINT` field, which is empty by default.

**Process**: ① Log in to HF and load → ② Select competition/trajectory → ③ Select breakpoint on the score curve or table → ④ Generate → ⑤ Compare. Results are automatically appended to `results.jsonl`.

## 1. Configuration

In [ ]:
!pip install plotly

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os, json, re, glob, difflib, textwrap, datetime, hashlib
from pathlib import Path
import pandas as pd
import ipywidgets as W
from IPython.display import display, HTML, clear_output

from google.colab import userdata
os.environ["LITELLM_API_KEY"] = userdata.get("LITELLM_API_KEY")

CFG = dict(
    # ---- Data Source ----
    DATA_SOURCE  = "hf",                  # "hf": Load from Hugging Face (requires login); "local": Use local paths below
    HF_REPO      = "jerryyan/TraceML",
    HF_SPLIT     = "paired",              # "paired" | "humans_only" | "experiment_run"
    # WORK_DIR     = "/content/drive/MyDrive/traceml_work",
    WORK_DIR     = "/content/traceml_work",
    # Used in local mode:
    ACTION_PARQUET = "TraceML/data/paired/action.parquet",
    NOTEBOOK_DIR   = "TraceML/trajectories_human",
    COMPETITIONS   = "TraceML/manifests/competitions.json",

    TASK_DESC_DIR  = "task_descriptions",   # Optional: <comp>.md, competition description for the model (suggested: overview+data description from Kaggle page)
    # RESULTS_PATH   = "/content/drive/MyDrive/traceml_work/results.jsonl",
    RESULTS_PATH   = "/content/traceml_work/results.jsonl",

    # ---- Model ----
    PROVIDER   = "proxy",               # "proxy" | "mock"
    MODEL      = "wine-claude-opus-4-6",               # Model name available on the proxy
    API_BASE   = "https://ai-gateway.andrew.cmu.edu",
    API_KEY    = os.environ.get("LITELLM_API_KEY", "your_api_key"),
    MAX_TOKENS = 2000,
    N_SAMPLES  = 1,
    TEMPERATURE= 0.7,
    JUDGE_MODEL   = "wine-claude-sonnet-4-6",   # Name on the proxy
    JUDGE_ENABLED = True,

    # ---- Context for Model ----
    HISTORY_CODE      = "diffs",      # How to provide snapshots for each step: "diffs" (actual diffs between adjacent versions) | "full" (full code for each version, long) | "none"
    HISTORY_REASONING = True,         # Whether to provide reasoning between steps (TraceML annotated goal_nl + diff_summary)
    HISTORY_MAX_CHARS = 80000,        # Maximum total length of history, oldest transitions discarded if exceeded
    DIFF_MAX_LINES    = 400,          # Maximum number of lines for a single diff
    CODE_MAX_CHARS    = 40000,        # Truncation length for breakpoint code
    HIDE_SCORES       = False,        # If True, historical scores are not shown to the model
    HINT              = "",           # Knowledge injected at the breakpoint (tacit knowledge variable). Default empty = no injection.
)

## 2. Load Data

In [ ]:
import tarfile

def load_actions_df(df):
    for c in ["v_old","v_new"]:
        df[c] = df[c].astype(int)
    df["key_id"] = df["key_id"].astype(str)
    if "is_agent" not in df: df["is_agent"] = False
    # Only keep version edges within the same kernel; fork/code_sim edges cross kernels and do not form a single trajectory.
    if "edge_kind" in df:
        df = df[df["edge_kind"].fillna("version") == "version"]
    return df.sort_values(["comp","key_id","v_old"]).reset_index(drop=True)

os.makedirs(CFG["WORK_DIR"], exist_ok=True)

if CFG["DATA_SOURCE"] == "hf":
    from huggingface_hub import notebook_login, hf_hub_download, get_token
    from datasets import load_dataset
    if get_token() is None:
        notebook_login()
        assert get_token() is not None, "Re-run this cell after logging in."
    act_path = hf_hub_download(CFG["HF_REPO"], f"data/{CFG['HF_SPLIT']}/action.parquet", repo_type="dataset")
    ACT = load_actions_df(pd.read_parquet(act_path))
    comp_json = hf_hub_download(CFG["HF_REPO"], "manifests/competitions.json", repo_type="dataset")
    COMPS = json.load(open(comp_json))
    # Source code: download tar.gz (2.9 GB) and extract once
    NB_ROOT = os.path.join(CFG["WORK_DIR"], "trajectories_human")
    if not os.path.isdir(NB_ROOT) or not glob.glob(os.path.join(NB_ROOT, "**", "*.ipynb"), recursive=True):
        tar_path = hf_hub_download(CFG["HF_REPO"], "trajectories_human.tar.gz", repo_type="dataset")
        print("extracting", tar_path, "->", NB_ROOT)
        with tarfile.open(tar_path) as tf:
            tf.extractall(NB_ROOT)
    CFG["NOTEBOOK_DIR"] = NB_ROOT
else:
    ACT = load_actions_df(pd.read_parquet(CFG["ACTION_PARQUET"]))
    COMPS = json.load(open(CFG["COMPETITIONS"])) if CFG["COMPETITIONS"] and os.path.exists(CFG["COMPETITIONS"]) else {}

print(f"transitions: {len(ACT):,}  |  trajectories: {ACT.key_id.nunique():,}  |  comps: {ACT.comp.nunique()}")
print("columns:", list(ACT.columns))

data/paired/action.parquet: reconstructing file:   0%|          |  0.00B / 4.45MB            

data/paired/action.parquet: downloading bytes:           |  0.00B            

competitions.json:   0%|          | 0.00/53.1k [00:00<?, ?B/s]

trajectories_human.tar.gz: reconstructing file:   0%|          |  0.00B / 2.96GB            

trajectories_human.tar.gz: downloading bytes:           |  0.00B            

extracting /root/.cache/huggingface/hub/datasets--jerryyan--TraceML/snapshots/9a3c790c2ad7c77bf5f56a851a72fc48b87afc4f/trajectories_human.tar.gz -> /content/traceml_work/trajectories_human


/tmp/ipykernel_1211/162407465.py:31: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(NB_ROOT)


transitions: 14,576  |  trajectories: 630  |  comps: 7
columns: ['key_id', 'comp', 'group', 'v_old', 'v_new', 'model', 'coarse_actions', 'fine_actions', 'intents', 'magnitude', 'score_effect', 'goal_nl', 'diff_summary', 'score_old', 'score_new', 'orig_v_old', 'orig_v_new', 'depth_old', 'depth_new', 'stage_old', 'stage_new', 'is_best_branch', 'is_agent', 'edge_kind', 'edge_kind_label', 'parent_node_id', 'child_node_id', 'parent_kernel_id', 'child_kernel_id', 'tree_id', 'ctime_old', 'ctime_new']


### 2.1 Build Index: Version → Source Code

The unzipped directory is organized as `kaggle_kernels/<kernel_id>/...`, and files might be named `<KernelVersionId>.ipynb` or `v<k>.ipynb`. After running the next cell, check the "indexed notebooks" counts; if both are 0, `print(glob.glob(...)[:5])` to see the actual path and adjust. The indexer below builds an index by two keys: `node_id` (`h:<KernelVersionId>`, from `parent_node_id`/`child_node_id`) and `(kernel_id, version_no)`. If your unzipped directory structure is different, modify `index_notebooks` accordingly.

In [ ]:
def nb_to_source(path, keep_markdown=True):
    """Converts .ipynb to a plain text code segment (outputs are already stripped in the dataset)."""
    try:
        nb = json.load(open(path, encoding="utf-8"))
    except Exception as e:
        return f"# [failed to read notebook: {e}]"
    parts = []
    for c in nb.get("cells", []):
        src = "".join(c.get("source", []))
        if c.get("cell_type") == "code":
            parts.append(src)
        elif keep_markdown and c.get("cell_type") == "markdown":
            parts.append("\n".join("# " + l for l in src.splitlines()))
    return "\n\n# ---CELL---\n\n".join(parts)

def index_notebooks(root):
    """Returns {node_id: path} and {(kernel_id, version_no): path}."""
    by_node, by_kv = {}, {}
    if not root or not os.path.isdir(root):
        print("NOTEBOOK_DIR does not exist; will only use label summaries as breakpoint state.")
        return by_node, by_kv
    for p in glob.glob(os.path.join(root, "**", "*.ipynb"), recursive=True):
        p = Path(p)
        m = re.match(r"^v?(\d+)$", p.stem)
        if not m:
            continue
        # Structure: .../<kernel_id>/versions/v019.ipynb -> kernel_id is two levels up; compatible with .../<kernel_id>/v019.ipynb
        kernel = p.parent.parent.name if p.parent.name == "versions" else p.parent.name
        if kernel.isdigit():
            by_kv[(kernel, int(m.group(1)))] = str(p)
    print(f"indexed notebooks: by_node={len(by_node):,}  by_(kernel,version)={len(by_kv):,}")
    return by_node, by_kv

NB_BY_NODE, NB_BY_KV = index_notebooks(CFG["NOTEBOOK_DIR"])

def source_at(row_or_kernel, version=None, which="old"):
    """Retrieves source code for a specific version. Pass action row + which∈{old,new}, or pass (kernel_id, version)."""
    if version is None:
        r = row_or_kernel
        g = (lambda c: getattr(r, c)) if not isinstance(r, (pd.Series, dict)) else (lambda c: r[c])
        node = g("parent_node_id" if which=="old" else "child_node_id")
        v    = g("v_old") if which=="old" else g("v_new")
        k    = str(g("key_id"))
    else:
        node, k, v = None, str(row_or_kernel), int(version)
    p = NB_BY_NODE.get(node) if node else None
    p = p or NB_BY_KV.get((k, v))
    return nb_to_source(p) if p else None

indexed notebooks: by_node=0  by_(kernel,version)=167,410


## 4. Assemble Context for the Model

In [ ]:
def load_task_desc(comp):
    p = os.path.join(CFG["TASK_DESC_DIR"], f"{comp}.md")
    if os.path.exists(p):
        return open(p, encoding="utf-8").read()
    m = COMPS.get(comp, {})
    return f"{m.get('name', comp)} (Kaggle). Metric: {m.get('metric', 'unknown')}, {m.get('score_direction', '')} is better."

def fmt_score(x):
    return "n/a" if pd.isna(x) else f"{x:.5f}"

def _diff(a, b, la, lb, max_lines, n=2):
    d = list(difflib.unified_diff(a.splitlines(), b.splitlines(), la, lb, lineterm="", n=n))
    if len(d) > max_lines:
        d = d[:max_lines] + [f"... [{len(d)-max_lines} more diff lines omitted]"]
    return "\n".join(d) if d else "(no code change in this version)"

def _score_line(r, prev_scored, best, direction):
    """Current version score, change relative to the last scored version, change relative to the current best score."""
    if pd.isna(r.score_new):
        return "leaderboard: not submitted"
    s = f"leaderboard: {r.score_new:.5f}"
    if prev_scored is not None:
        d = r.score_new - prev_scored
        better = (d > 0) if direction == "max" else (d < 0)
        s += f"  (Δ vs previous scored version: {d:+.5f}, {'better' if better else 'worse' if d != 0 else 'same'})"
    if best is not None:
        d = r.score_new - best
        s += f"  (Δ vs best so far: {d:+.5f})"
    return s

def build_history(t, bp, cfg):
    """History of v_1..v_t: snapshot changes for each step + reasoning between steps (annotations) + score and relative changes."""
    hist = t[t.v_new <= bp]
    if len(hist) == 0 or (cfg["HISTORY_CODE"] == "none" and not cfg["HISTORY_REASONING"]):
        return "(no history provided)"
    direction = score_direction(t.comp.iloc[0])
    blocks, prev_scored, best = [], None, None
    first = hist.iloc[0]
    if not cfg["HIDE_SCORES"]:
        blocks.append(f"Metric direction: {'higher' if direction=='max' else 'lower'} is better. "
                      f"Leaderboard at v{first.v_old} (before the edits below): {fmt_score(first.score_old)}")
        if not pd.isna(first.score_old):
            prev_scored = best = first.score_old
    for r in hist.itertuples():
        lines = [f"## v{r.v_old} -> v{r.v_new}"]
        if cfg["HISTORY_REASONING"]:
            lines.append(f"Reasoning: {r.goal_nl}")
            lines.append(f"Change: {r.diff_summary}")
        if cfg["HISTORY_CODE"] == "full":
            b = source_at(r, which="new")
            lines.append("```python\n" + (b if b is not None else "(source not available)") + "\n```")
        elif cfg["HISTORY_CODE"] == "diffs":
            a, b = source_at(r, which="old"), source_at(r, which="new")
            lines.append("```diff\n" + _diff(a, b, f"v{r.v_old}", f"v{r.v_new}", cfg["DIFF_MAX_LINES"]) + "\n```"
                         if a is not None and b is not None else "(source not available for this transition)")
        if not cfg["HIDE_SCORES"]:
            lines.append(_score_line(r, prev_scored, best, direction))
            if not pd.isna(r.score_new):
                prev_scored = r.score_new
                best = r.score_new if best is None else (max(best, r.score_new) if direction == "max" else min(best, r.score_new))
        blocks.append("\n".join(lines))
    # If too long, discard from the earliest transitions (keeping the initial score explanation).
    dropped = 0
    body = blocks[1:] if not cfg["HIDE_SCORES"] else blocks
    while len(body) > 1 and sum(len(x) for x in body) > cfg["HISTORY_MAX_CHARS"]:
        body.pop(0); dropped += 1
    if dropped:
        first_kept_v = hist.iloc[dropped].v_old
        body.insert(0, f"(the first {dropped} transition(s) are omitted for length; the code at v{first_kept_v} is the result of those edits)")
    return "\n\n".join(([blocks[0]] if not cfg["HIDE_SCORES"] else []) + body)

SYSTEM_PROMPT = """You are an experienced Kaggle competitor continuing someone else's notebook mid-competition.
You are given: the competition description, the edit history so far (for each past version: the author's reasoning, the code change, and the public leaderboard score with its change relative to the previous submission), and the current full code. Notebook outputs are not available.
Decide the single next edit you would make and explain what in the current state motivates it.
Respond ONLY with JSON (no markdown fences) with these keys:
{
  "observations": ["<things you noticed in the current code/data/scores that matter for the decision>"],
  "diagnosis": "<what you believe is the current bottleneck or risk>",
  "next_action_category": ["<one or more of: data, features, augmentation, model, training, validation, inference, ensemble, infra, housekeeping>"],
  "next_edit": "<concrete description of the change, at the level of a commit message>",
  "why_this_over_alternatives": "<what alternatives you considered and why not>",
  "expected_effect": "<what you expect to happen to validation/leaderboard score and why>",
  "code_sketch": "<optional: minimal code snippet or diff of the change>"
}"""

def build_user_prompt(comp, t, bp, cfg):
    code_now = None
    row_at_bp = t[t.v_old == bp]
    if len(row_at_bp):
        code_now = source_at(row_at_bp.iloc[0], which="old")
    if code_now is None:
        code_now = "(source code for this version is not available; rely on the history summary)"
    if len(code_now) > cfg["CODE_MAX_CHARS"]:
        code_now = code_now[:cfg["CODE_MAX_CHARS"]] + f"\n# ... [truncated, {len(code_now)-cfg['CODE_MAX_CHARS']} chars omitted]"
    hist = build_history(t, bp, cfg)
    hint = cfg.get("HINT", "").strip()
    hint_block = f"\n# Note from a collaborator\n{hint}\n" if hint else ""
    return f"""# Competition
{load_task_desc(comp)}

# Edit history so far (oldest first)
{hist}

# Current code (version v{bp})
```python
{code_now}
```
{hint_block}
Decide your next edit."""

## 5. Model Invocation

In [ ]:
def call_model(system, user, cfg, model=None):
    if cfg["PROVIDER"] == "mock":
        return json.dumps({"observations":["(mock)"],"diagnosis":"(mock)","next_action_category":["model"],
                           "next_edit":"(mock) swap backbone","why_this_over_alternatives":"(mock)",
                           "expected_effect":"(mock)","code_sketch":""})
    import openai
    client = openai.OpenAI(api_key=cfg["API_KEY"], base_url=cfg["API_BASE"])
    r = client.chat.completions.create(
        model=model or cfg["MODEL"],
        messages=[{"role":"system","content":system},{"role":"user","content":user}],
        max_tokens=cfg["MAX_TOKENS"],
        temperature=cfg["TEMPERATURE"],
    )
    return r.choices[0].message.content

def parse_json(txt):
    txt = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    try:
        return json.loads(txt)
    except Exception:
        m = re.search(r"\{.*\}", txt, flags=re.S)
        if m:
            try: return json.loads(m.group(0))
            except Exception: pass
    return {"_raw": txt}

def generate_at_breakpoint(key_id, bp, cfg=CFG, n=None):
    t = trajectory(key_id); comp = t.comp.iloc[0]
    user = build_user_prompt(comp, t, bp, cfg)
    outs = []
    for i in range(n or cfg["N_SAMPLES"]):
        raw = call_model(SYSTEM_PROMPT, user, cfg)
        outs.append(parse_json(raw))
    return dict(key_id=str(key_id), comp=comp, breakpoint=int(bp), prompt=user, outputs=outs,
                cfg={k:cfg[k] for k in ("PROVIDER","MODEL","HISTORY_CODE","HISTORY_REASONING","HISTORY_MAX_CHARS","HIDE_SCORES","CODE_MAX_CHARS","TEMPERATURE","HINT")},
                ts=datetime.datetime.now().isoformat(timespec="seconds"))


## 6. Comparison

In [ ]:
def expert_step(t, bp, k_ahead=3):
    """The expert's actual transition at the breakpoint, and an overview of the next k steps."""
    nxt = t[t.v_old == bp]
    if not len(nxt): return None, t.iloc[0:0]
    r = nxt.iloc[0]
    later = t[(t.v_old > bp)].head(k_ahead)
    return r, later

def expert_diff(r, max_lines=300):
    a, b = source_at(r, which="old"), source_at(r, which="new")
    if a is None or b is None: return "(source not available for one of the versions)"
    d = list(difflib.unified_diff(a.splitlines(), b.splitlines(), f"v{r.v_old}", f"v{r.v_new}", lineterm="", n=2))
    if len(d) > max_lines: d = d[:max_lines] + [f"... [{len(d)-max_lines} more lines]"]
    return "\n".join(d) or "(no code change)"

def _esc(s):
    import html; return html.escape(str(s))

def _list(x):
    if isinstance(x, str):
        try: x = json.loads(x)
        except Exception: return _esc(x)
    if isinstance(x, list):
        return ", ".join(_esc(i if not isinstance(i,dict) else i.get("action", i)) for i in x)
    return _esc(x)

def render_comparison(res):
    t = trajectory(res["key_id"]); bp = res["breakpoint"]
    r, later = expert_step(t, bp)
    exp_html = "<i>The breakpoint is the last version, so there is no expert next step.</i>"
    if r is not None:
        eff = f"{fmt_score(r.score_old)} → {fmt_score(r.score_new)} ({r.score_effect})"
        later_html = "".join(f"<li>v{x.v_old}→v{x.v_new} [{_list(x.coarse_actions)}] {_esc(x.goal_nl)} ({x.score_effect})</li>" for x in later.itertuples())
        exp_html = f"""
        <p><b>v{r.v_old} → v{r.v_new}</b> &nbsp; size=<b>{r.magnitude}</b> &nbsp; score {eff}</p>
        <p><b>coarse:</b> {_list(r.coarse_actions)}<br><b>fine:</b> {_list(r.fine_actions)}<br><b>intent:</b> {_list(r.intents)}</p>
        <p><b>goal (label):</b> {_esc(r.goal_nl)}</p>
        <p><b>diff summary (label):</b> {_esc(r.diff_summary)}</p>
        <details><summary>Real code diff</summary><pre style='font-size:11px;max-height:400px;overflow:auto'>{_esc(expert_diff(r))}</pre></details>
        <p><b>Next steps:</b></p><ul style='font-size:12px'>{later_html}</ul>"""

    model_html = ""
    exp_cats = set(json.loads(r.coarse_actions)) if r is not None and isinstance(r.coarse_actions,str) else set()
    for i, o in enumerate(res["outputs"]):
        if "_raw" in o:
            model_html += f"<h4>sample {i+1}</h4><pre>{_esc(o['_raw'])}</pre>"; continue
        cats = set(o.get("next_action_category", []) or [])
        overlap = cats & exp_cats
        badge = f"<span style='background:{'#d4edda' if overlap else '#f8d7da'};padding:2px 6px;border-radius:4px'>coarse overlap: {', '.join(sorted(overlap)) or 'none'}</span>"
        obs = "".join(f"<li>{_esc(x)}</li>" for x in o.get("observations", []))
        model_html += f"""
        <h4>sample {i+1} &nbsp; {badge}</h4>
        <p><b>observations:</b><ul style='font-size:12px'>{obs}</ul></p>
        <p><b>diagnosis:</b> {_esc(o.get('diagnosis',''))}</p>
        <p><b>category:</b> {_list(o.get('next_action_category',[]))}</p>
        <p><b>next edit:</b> {_esc(o.get('next_edit',''))}</p>
        <p><b>why over alternatives:</b> {_esc(o.get('why_this_over_alternatives',''))}</p>
        <p><b>expected effect:</b> {_esc(o.get('expected_effect',''))}</p>
        <details><summary>code sketch</summary><pre style='font-size:11px'>{_esc(o.get('code_sketch',''))}</pre></details>"""

    html = f"""
    <div style='display:flex;gap:16px;font-size:13px'>
      <div style='flex:1;border:1px solid #ccc;padding:10px;border-radius:6px'><h3>What the expert did next</h3>{exp_html}</div>
      <div style='flex:1;border:1px solid #ccc;padding:10px;border-radius:6px'><h3>What the model will do ({CFG["MODEL"]})</h3>{model_html}</div>
    </div>
    <details style='margin-top:8px'><summary>Full prompt（{len(res['prompt']):,} chars）</summary><pre style='font-size:10px;max-height:500px;overflow:auto'>{_esc(res['prompt'])}</pre></details>
    """
    return HTML(html)

def save_result(res, path=CFG["RESULTS_PATH"]):
    r_exp, _ = expert_step(trajectory(res["key_id"]), res["breakpoint"])
    rec = dict(res)
    rec["expert"] = None if r_exp is None else dict(
        v_old=int(r_exp.v_old), v_new=int(r_exp.v_new), coarse_actions=r_exp.coarse_actions, fine_actions=r_exp.fine_actions,
        intents=r_exp.intents, magnitude=r_exp.magnitude, score_effect=r_exp.score_effect,
        score_old=None if pd.isna(r_exp.score_old) else float(r_exp.score_old),
        score_new=None if pd.isna(r_exp.score_new) else float(r_exp.score_new),
        goal_nl=r_exp.goal_nl, diff_summary=r_exp.diff_summary)
    rec["prompt_sha1"] = hashlib.sha1(res["prompt"].encode()).hexdigest()
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [ ]:
JUDGE_SYSTEM = """You are evaluating whether a model's proposed next step on a Kaggle notebook matches what a human expert actually did at the same point.
You will see the expert's real code diff and score change, and the model's proposed edit with its stated observations.
Judge on two separate dimensions, then give an overall verdict. Respond ONLY with JSON:
{
  "action_match": "same" | "partial" | "different",
  "action_note": "<how the model's concrete edit relates to the expert's real diff; compare substance, not category labels>",
  "observation_match": "captured" | "partial" | "missed",
  "observation_note": "<did the model's observations/diagnosis include the condition that makes the expert's edit sensible? what did it notice or miss?>",
  "expert_implicit_condition": "<your best reconstruction of what the expert must have noticed or known to make this edit, stated as: when <condition>, do <action>>",
  "overall": 0 | 1 | 2 | 3,
  "explanation": "<3-6 sentences, in English>"
}
Scale: 0 = unrelated; 1 = same general area but different move; 2 = same move, different or missing rationale; 3 = same move for the same reason."""

def build_judge_prompt(res, sample):
    t = trajectory(res["key_id"]); r, later = expert_step(t, res["breakpoint"])
    exp = f"""# Expert's actual transition v{r.v_old} -> v{r.v_new}
Annotated goal: {r.goal_nl}
Annotated change: {r.diff_summary}
Score: {fmt_score(r.score_old)} -> {fmt_score(r.score_new)} ({r.score_effect})
Real diff:
```diff
{expert_diff(r, max_lines=400)}
```
Expert's following steps (for context only): {'; '.join(f'v{x.v_old}->v{x.v_new}: {x.goal_nl}' for x in later.itertuples())}"""
    mdl = json.dumps(sample, ensure_ascii=False, indent=1)
    return f"{exp}\n\n# Model's proposed next step\n{mdl}\n\nJudge."

def judge_result(res, cfg=CFG):
    r, _ = expert_step(trajectory(res["key_id"]), res["breakpoint"])
    if r is None:
        res["judgements"] = []; return res
    res["judgements"] = []
    for o in res["outputs"]:
        raw = call_model(JUDGE_SYSTEM, build_judge_prompt(res, o), cfg, model=cfg["JUDGE_MODEL"])
        res["judgements"].append(parse_json(raw))
    res["judge_model"] = cfg["JUDGE_MODEL"]
    return res

def render_judgement(res):
    js = res.get("judgements") or []
    if not js: return HTML("")
    color = {0:"#f8d7da",1:"#ffe5b4",2:"#fff3cd",3:"#d4edda"}
    blocks = []
    for i, j in enumerate(js):
        if "_raw" in j:
            blocks.append(f"<h4>sample {i+1}</h4><pre>{_esc(j['_raw'])}</pre>"); continue
        ov = j.get("overall", "?")
        blocks.append(f"""
        <div style='border-left:6px solid {color.get(ov,'#ccc')};padding:8px 12px;margin:6px 0;font-size:13px'>
          <b>sample {i+1} — overall {ov}/3</b> &nbsp;
          action: <b>{_esc(j.get('action_match'))}</b> &nbsp; observation: <b>{_esc(j.get('observation_match'))}</b>
          <p><b>action:</b> {_esc(j.get('action_note'))}</p>
          <p><b>observation:</b> {_esc(j.get('observation_note'))}</p>
          <p><b>Expert implicit condition (reconstructed by judge):</b> {_esc(j.get('expert_implicit_condition'))}</p>
          <p>{_esc(j.get('explanation'))}</p>
        </div>""")
    return HTML(f"<h3>Judge ({_esc(res.get('judge_model'))})</h3>{''.join(blocks)}")



## Heuristic: what's left is what's valuable



In [ ]:
def trajectory(key_id):
    t = ACT[ACT.key_id == str(key_id)].sort_values("v_old").reset_index(drop=True)
    return t

def traj_summary_table():
    g = ACT[~ACT.is_agent].groupby(["comp","key_id","group"], dropna=False)
    s = g.agg(n_trans=("v_old","size"),
              n_scored=("score_new", lambda x: x.notna().sum()),
              best=("score_new","max"),
              first_v=("v_old","min"), last_v=("v_new","max")).reset_index()
    return s

TRAJ = traj_summary_table()

def score_direction(comp):
    """Reads score direction from competitions.json; if not found, assumes higher is better."""
    info = COMPS.get(comp) if isinstance(COMPS, dict) else None
    if isinstance(info, dict):
        for k in ("direction","score_direction","higher_is_better","maximize"):
            if k in info:
                v = info[k]
                if isinstance(v,bool): return "max" if v else "min"
                return "max" if str(v).lower() in ("max","maximize","higher","up") else "min"
    return "max"

def render_traj_table(t, breakpoint=None):
    rows = []
    for i, r in t.iterrows():
        mark = "▶" if breakpoint is not None and r.v_old == breakpoint else ""
        acts = ", ".join(json.loads(r.coarse_actions)) if isinstance(r.coarse_actions,str) else r.coarse_actions
        rows.append(f"<tr style='{'background:#fff3cd' if mark else ''}'>"
                    f"<td>{mark}</td><td>{r.v_old}→{r.v_new}</td>"
                    f"<td>{'' if pd.isna(r.score_old) else round(r.score_old,5)}</td>"
                    f"<td>{'' if pd.isna(r.score_new) else round(r.score_new,5)}</td>"
                    f"<td>{r.score_effect}</td><td>{r.magnitude}</td><td>{acts}</td>"
                    f"<td style='max-width:520px'>{r.goal_nl}</td></tr>")
    head = "<tr><th></th><th>v</th><th>score_old</th><th>score_new</th><th>effect</th><th>size</th><th>coarse actions</th><th>goal (label)</th></tr>"
    return HTML(f"<table style='font-size:12px;border-collapse:collapse'>{head}{''.join(rows)}</table>")

In [ ]:
# ====================== Pivot pre-annotation: retention filter -> TK matching -> pivot table ======================
# Requires from earlier cells: CFG, ACT, TRAJ, trajectory(), source_at(), build_history(), expert_diff(),
# load_task_desc(), fmt_score(), call_model(system, user, cfg, model=None), parse_json(), _esc().
# Requires CFG["JUDGE_MODEL"] and CFG["WORK_DIR"].
import concurrent.futures as cf

ANNOT_PATH  = os.path.join(CFG["WORK_DIR"], "annotations.jsonl")     # every annotated transition (raw)
PIVOT_PATH  = os.path.join(CFG["WORK_DIR"], "pivots.csv")            # transitions that matched a TK item (read by Section 3)

# ---------------------------------------------------------------- 1. Retention filter ----------------------------------------------------------------
def _norm_lines(src):
    out = []
    for l in (src or "").splitlines():
        l = re.sub(r"#.*$", "", l).strip()
        if len(l) >= 8:
            out.append(l)
    return out

def best_version_row(t):
    """Row whose v_new is the best-scoring version (direction-aware). Falls back to the last row if nothing was scored."""
    scored = t[t.score_new.notna()]
    if not len(scored):
        return t.iloc[-1]
    asc = score_direction(t.comp.iloc[0]) == "min"
    return scored.sort_values(["score_new", "v_new"], ascending=[asc, True]).iloc[0]   # ties -> earliest

def retention_for_trajectory(t):
    """For each transition up to the best-scoring version, how many of its added code lines still exist in that version.
    Transitions after the best version are dropped."""
    best = best_version_row(t)
    t = t[t.v_new <= best.v_new].reset_index(drop=True)
    ref_src = source_at(best, which="new")
    ref_set = set(_norm_lines(ref_src)) if ref_src else set()
    rows = []
    for r in t.itertuples():
        a, b = source_at(r, which="old"), source_at(r, which="new")
        if a is None or b is None or not ref_set:
            rows.append((r.v_old, None, None, None)); continue
        added = [l[1:] for l in difflib.unified_diff(a.splitlines(), b.splitlines(), lineterm="", n=0)
                 if l.startswith("+") and not l.startswith("+++")]
        added = set(_norm_lines("\n".join(added)))
        kept = added & ref_set
        rows.append((r.v_old, len(added), len(kept), (len(kept) / len(added)) if added else None))
    df = pd.DataFrame(rows, columns=["v_old", "added_lines", "retained_lines", "retained_ratio"])
    out = t.merge(df, on="v_old")
    out.attrs["best_v_new"] = int(best.v_new)
    return out

def retained_transitions(t):
    """Full trajectory with retention columns; `keep` marks transitions whose added code survives to the final version."""
    tr = retention_for_trajectory(t)
    tr["keep"] = tr.retained_lines.fillna(0) >= 1
    return tr

# ---------------------------------------------------------------- 2. Tacit-knowledge list ------------------------------------------------------------
TACIT_KNOWLEDGE = [
  {"id": "TK01_validation_probing",
   "context": "The notebook uses a default split (random or stratified K-fold), and the data has a visible grouping or ordering structure (an entity id repeated across rows, a timestamp, a source/site column) that the split does not respect.",
   "rule": "If test rows come from unseen groups or later time, rebuild the split to mirror that (GroupKFold on the entity, time-based holdout); else keep the default split. Treat the CV scheme as a hypothesis about how test differs from train, not as a fixed setting."},

  {"id": "TK02_cv_lb_gap_diagnosis",
   "context": "Over several submissions, local CV and public LB move in different directions, or the CV-LB gap is far larger than the differences between models.",
   "rule": "If the gap persists over 2-3 submissions, stop tuning models and look for the cause in the split or a leaking feature; else treat a single LB move as noise and keep trusting CV."},

  {"id": "TK03_leak_artifacts",
   "context": "A feature has implausibly high importance or predictive power, and it looks like an artifact: an id that increases with time, a row index, file order, float precision or missingness patterns that differ between train and test.",
   "rule": "If the strongest signal comes from such an artifact, treat the CV score as leaked and remove the feature before doing anything else; else proceed with feature work on the real signal."},

  {"id": "TK04_adversarial_shift_check",
   "context": "Train and test may not come from the same distribution (different time range, different sources), and a train-vs-test classifier can be built.",
   "rule": "If a classifier separates train from test well above chance, find the drifting features and drop, coarsen, or reweight them before tuning; else assume i.i.d. and spend effort on model capacity."},

  {"id": "TK05_metric_aligned_objective",
   "context": "The competition metric is not the training loss and behaves differently from it (a thresholded metric like F1 or QWK, a ranking metric like MAP@K, an asymmetric or weighted custom score) and the model is trained on plain logloss or MSE.",
   "rule": "If the metric can be directly optimized in post-processing (thresholds, cut points, per-group rank normalization), fit that step on out-of-fold predictions; if not, change the loss or the early-stopping criterion to a surrogate of the metric; else train on the default loss."},

  {"id": "TK06_ensemble_orthogonality",
   "context": "An ensemble is being built and the candidate models are variants of the same family on the same features (many seeds or parameter settings of one GBDT).",
   "rule": "If the models' errors are highly correlated, add a model with a different representation (a neural net on raw sequences, a linear model on interactions, a GBDT on aggregates) rather than another variant, and set weights by out-of-fold prediction covariance; else simple averaging of variants is fine."},

  {"id": "TK07_residual_guided_features",
   "context": "The model has plateaued and the remaining gains from importance-ranked feature engineering are small.",
   "rule": "If the plateau persists, inspect the rows where out-of-fold error is largest and derive features from what those rows share (an interaction, an edge case, a subgroup); else continue with importance-guided feature work."},

  {"id": "TK08_capacity_suppression",
   "context": "The signal-to-noise ratio is low: small data, weak or noisy labels, per-fold scores vary widely, and hyperparameter search keeps finding new 'best' settings on validation.",
   "rule": "If the improvements being chased are smaller than the fold-to-fold spread, reduce capacity (shallower trees, more regularization, more subsampling) and prefer stable settings over the single best CV run; else let capacity and tuning run."},

  {"id": "TK09_lb_noise_floor",
   "context": "The public LB is computed on a small fraction of the test set, so a change smaller than its sampling noise is uninformative.",
   "rule": "If a change moves the LB by less than that noise, do not keep or revert on that basis and stop micro-tuning weights; else treat the move as evidence."},

  {"id": "TK10_late_stage_allocation",
   "context": "Little competition time or compute remains and the current model is already competitive.",
   "rule": "If the budget is nearly exhausted, stop adding new architectures or features and spend the remainder on seed averaging, fold ensembling, and pseudo-labeling; else structurally new ideas are still worth trying."},
]
TK_TEXT = "\n".join(f"- [{k['id']}] Context: {k['context']}\n  Rule: {k['rule']}" for k in TACIT_KNOWLEDGE)

ANNOT_SYSTEM = f"""You are annotating one edit made by a human expert in a Kaggle notebook.
You see the competition, the history of edits and public scores up to this point, the real code diff of this edit, and (for judging delayed effects only) what happened afterwards.
Decide whether this edit is an instance showing the expert is using one of the tacit-knowledge items below. Be strict: an item applies only if the edit is the item's Rule AND there is evidence that the item's Context actually held. An edit that merely resembles the action is not enough unless the context is visible.

{TK_TEXT}

Respond ONLY with JSON:
{{
  "tk_ids": ["<id>", ...],
  "evidence_type": "observed_in_code" | "observed_in_scores" | "inferred_from_action" | "none",
  "evidence": "<what in the diff, prior code, or score history shows the Context held; quote specifics>",
  "confidence": 0 | 1 | 2 | 3,
  "implicit_condition": "<the judgement the expert must have made, as: when <condition>, do <action>>",
  "novice_alternative": "<what a less experienced person would plausibly have done here instead; 'same' if no difference>",
  "is_pivot": true | false,
  "other_tacit_candidate": "<if the edit reflects an expert judgement NOT covered by any listed item, state it as: when <condition>, do <action>; else empty string>",
  "note": "<1-3 sentences>"
}}
Rules for the fields:
- If no item applies, return ONLY: {{"tk_ids": [], "other_tacit_candidate": "<...or empty string>", "note": "<one sentence>"}}. Omit all other keys.
- If at least one item applies, return all keys.
- is_pivot is about tacit judgement, not debugging skill: a bug fix is not a pivot unless a listed item explains why the expert looked there.
- confidence 3 = context clearly visible and action matches the rule; 2 = context visible but partial or indirect; 1 = action matches but context only inferred; 0 = no match."""

def build_annot_prompt(t, r, cfg=CFG):
    hist_cfg = dict(cfg, HISTORY_CODE="none", HISTORY_REASONING=True, HIDE_SCORES=False)
    later = t[t.v_old > r.v_old].head(5)
    later_txt = "\n".join(f"- v{x.v_old}->v{x.v_new}: {x.goal_nl}  [LB {fmt_score(x.score_new)}]" for x in later.itertuples()) or "(none)"
    return f"""# Competition
{load_task_desc(t.comp.iloc[0])}

# History before this edit
{build_history(t, r.v_old, hist_cfg)}

# This edit: v{r.v_old} -> v{r.v_new}
Annotated goal: {r.goal_nl}
Annotated change: {r.diff_summary}
Size: {r.magnitude}   Score: {fmt_score(r.score_old)} -> {fmt_score(r.score_new)} ({r.score_effect})
Retention: {r.retained_lines} of {r.added_lines} added lines survive to the final version
```diff
{expert_diff(r, max_lines=cfg["DIFF_MAX_LINES"])}
```

# What happened next (for judging delayed effect only; the expert did not know this)
{later_txt}

Annotate."""

def annotate_transition(t, r, cfg=CFG):
    """Annotate one transition; returns a record, does not write to disk."""
    prompt = build_annot_prompt(t, r, cfg)
    try:
        a = parse_json(call_model(ANNOT_SYSTEM, prompt, cfg, model=cfg["JUDGE_MODEL"]))
    except Exception as e:
        a = {"tk_ids": [], "evidence_type": "error", "note": repr(e)}
    a = {"tk_ids": [], "evidence_type": "none", "confidence": 0, "evidence": "", "implicit_condition": "",
         "novice_alternative": "", "is_pivot": False, "other_tacit_candidate": "", "note": "", **a}
    return dict(key_id=str(t.key_id.iloc[0]), comp=t.comp.iloc[0], group=t.group.iloc[0],
                v_old=int(r.v_old), v_new=int(r.v_new), magnitude=r.magnitude, score_effect=r.score_effect,
                score_old=None if pd.isna(r.score_old) else float(r.score_old),
                score_new=None if pd.isna(r.score_new) else float(r.score_new),
                added_lines=r.added_lines, retained_lines=r.retained_lines,
                goal_nl=r.goal_nl, judge_model=cfg["JUDGE_MODEL"],
                ts=datetime.datetime.now().isoformat(timespec="seconds"), prompt_chars=len(prompt), **a)

# ---------------------------------------------------------------- 3. Storage and pivot table ---------------------------------------------------------
def _already_done(path=ANNOT_PATH):
    if not os.path.exists(path): return set()
    return {(d["key_id"], d["v_old"]) for d in (json.loads(l) for l in open(path, encoding="utf-8"))}

def load_annotations(path=ANNOT_PATH):
    if not os.path.exists(path): return pd.DataFrame()
    df = pd.DataFrame([json.loads(l) for l in open(path, encoding="utf-8")])
    df["tk_ids"] = df["tk_ids"].apply(lambda x: ",".join(x) if isinstance(x, list) else (x or ""))
    return df

def build_pivot_table(min_conf=1, path=PIVOT_PATH):
    """Transitions that matched at least one TK item. Written to PIVOT_PATH; Section 3 reads this file."""
    df = load_annotations()
    if not len(df):
        piv = pd.DataFrame(columns=["comp","key_id","group","v_old","v_new","tk_ids","confidence","evidence_type","is_pivot",
                                    "implicit_condition","novice_alternative","note"])
    else:
        m = (df.tk_ids != "") & (df.confidence.fillna(0) >= min_conf)
        cols = ["comp","key_id","group","v_old","v_new","magnitude","score_effect","tk_ids","confidence","evidence_type","is_pivot",
                "implicit_condition","novice_alternative","other_tacit_candidate","note"]
        piv = df[m][[c for c in cols if c in df.columns]].sort_values(["confidence","comp","key_id","v_old"],
                                                                      ascending=[False,True,True,True]).reset_index(drop=True)
    piv.to_csv(path, index=False)
    return piv

def load_pivots(path=PIVOT_PATH):
    return pd.read_csv(path, dtype={"key_id": str}) if os.path.exists(path) else pd.DataFrame(columns=["key_id","v_old","tk_ids","confidence"])

def render_annotation(rec, t=None):
    t = t if t is not None else trajectory(rec["key_id"])
    r = t[t.v_old == rec["v_old"]].iloc[0]
    color = {0:"#eee", 1:"#ffe5b4", 2:"#fff3cd", 3:"#d4edda"}.get(rec.get("confidence"), "#eee")
    return HTML(f"""
    <div style='border-left:6px solid {color};padding:8px 12px;margin:8px 0;font-size:13px'>
      <b>{rec['key_id']} v{rec['v_old']}→v{rec['v_new']}</b> &nbsp; size={rec['magnitude']} &nbsp;
      {fmt_score(r.score_old)}→{fmt_score(r.score_new)} ({rec['score_effect']}) &nbsp;
      retained {rec.get('retained_lines')}/{rec.get('added_lines')} &nbsp;
      <b>conf {rec.get('confidence')}</b> &nbsp; tk=<b>{_esc(rec.get('tk_ids'))}</b> &nbsp;
      evidence_type={_esc(rec.get('evidence_type'))} &nbsp; pivot={rec.get('is_pivot')}
      <p><b>annotated goal:</b> {_esc(rec['goal_nl'])}</p>
      <p><b>evidence:</b> {_esc(rec.get('evidence'))}</p>
      <p><b>implicit condition:</b> {_esc(rec.get('implicit_condition'))}</p>
      <p><b>novice alternative:</b> {_esc(rec.get('novice_alternative'))}</p>
      <p><b>other tacit candidate:</b> {_esc(rec.get('other_tacit_candidate'))}</p>
      <p>{_esc(rec.get('note'))}</p>
      <details><summary>diff</summary><pre style='font-size:11px;max-height:350px;overflow:auto'>{_esc(expert_diff(r))}</pre></details>
    </div>""")

# ---------------------------------------------------------------- 4. Smoke test and batch run ----------------------------------------------------------
def smoke_test(n=10, show_prompt_first=True):
    """Annotate the first n retained transitions (first few human trajectories), display each, no disk write."""
    keys = list(ACT[~ACT.is_agent].sort_values(["comp","key_id"]).key_id.unique())
    rows, k_i = [], 0
    while len(rows) < n and k_i < len(keys):
        tr = retained_transitions(trajectory(keys[k_i])); k_i += 1
        rows += [(tr, r) for r in tr[tr.keep].itertuples()]
    recs = []
    for i, (tr, r) in enumerate(rows[:n]):
        if show_prompt_first and i == 0:
            p = build_annot_prompt(tr, r)
            print(f"prompt for {tr.key_id.iloc[0]} v{r.v_old}->v{r.v_new}: {len(p):,} chars\n" + "-"*60); print(p[:3000]); print("-"*60)
        rec = annotate_transition(tr, r); recs.append(rec)
        display(render_annotation(rec, tr))
    return recs

def batch_annotate(key_ids, workers=4, path=ANNOT_PATH, progress=None):
    done = _already_done(path)
    jobs = []
    for k in key_ids:
        tr = retained_transitions(trajectory(k))
        jobs += [(tr, r) for r in tr[tr.keep].itertuples() if (str(k), int(r.v_old)) not in done]
    if progress is not None: progress.max = max(len(jobs), 1); progress.value = 0
    print(f"{len(key_ids)} trajectories, {len(jobs)} retained transitions to annotate")
    with cf.ThreadPoolExecutor(workers) as ex, open(path, "a", encoding="utf-8") as f:
        for rec in ex.map(lambda tr: annotate_transition(*tr), jobs):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
            if progress is not None: progress.value += 1
    ANN = load_annotations(path)
    ANN.to_csv(os.path.join(CFG["WORK_DIR"], "annotations.csv"), index=False)
    piv = build_pivot_table()
    print(f"done. annotations={len(ANN)}  pivots={len(piv)}  -> {PIVOT_PATH}")
    if len(ANN):
        print("\nTK frequency (conf>=1):\n" +
              ANN[ANN.confidence.fillna(0) >= 1].tk_ids.str.split(",").explode().replace("", pd.NA).dropna().value_counts().to_string())
        print("\nEvidence type:\n" + ANN.evidence_type.value_counts().to_string())
    return ANN, piv

# ---------------------------------------------------------------- 5. Widgets -------------------------------------------------------------------------
w_a_comps = W.SelectMultiple(options=sorted(TRAJ.comp.unique()), value=(), description="comps", rows=6, layout=W.Layout(width="60%"))
w_a_tiers = W.SelectMultiple(options=["Grandmaster","Master","Expert","Contributor"], value=("Grandmaster","Master"), description="tiers", rows=4)
w_a_max   = W.Dropdown(options=[1,2,3,5,10,20,1000], value=3, description="N per comp")
w_a_min   = W.Dropdown(options=[3,5,10,20], value=5, description="min trans")
w_a_work  = W.Dropdown(options=[1,2,4,8], value=4, description="workers")
w_a_go    = W.Button(description="Run pre-annotation", button_style="warning")
w_a_prog  = W.IntProgress(value=0, min=0, max=1, layout=W.Layout(width="90%"))
out_annot = W.Output()

def _targets():
    sub = TRAJ[(TRAJ.n_trans >= w_a_min.value) & (TRAJ.comp.isin(w_a_comps.value)) & TRAJ.best.notna()].copy()
    sub["rank_score"] = sub.apply(lambda r: r.best if score_direction(r.comp) == "max" else -r.best, axis=1)
    sub = sub.sort_values(["comp","rank_score"], ascending=[True, False])
    return list(sub.groupby("comp").head(w_a_max.value).key_id)

def _on_batch(_):
    with out_annot:
        clear_output(wait=True)
        ks = _targets()
        if not ks:
            print("No matching trajectories. Select at least one competition in 'comps'."); return
        ANN, piv = batch_annotate(ks, workers=w_a_work.value, progress=w_a_prog)
        pd.set_option("display.max_colwidth", 160); display(piv.head(30))

w_a_go.on_click(_on_batch)
display(W.VBox([W.HBox([w_a_comps]), W.HBox([w_a_max, w_a_min, w_a_work]), w_a_go, w_a_prog, out_annot]))

In [ ]:
comps = sorted(TRAJ.comp.unique())
w_comp = W.Dropdown(options=comps, description="comp", layout=W.Layout(width="60%"))
w_traj = W.Dropdown(description="trajectory", layout=W.Layout(width="90%"))
w_bp   = W.Dropdown(options=[None], description="breakpoint v_t", layout=W.Layout(width="60%"))
out_table = W.Output()

out_pivots = W.Output()

def render_pivot_cards(t, bp=None):
    piv = load_pivots(); piv = piv[piv.key_id == str(t.key_id.iloc[0])].sort_values("v_old")
    if not len(piv): return HTML("<i>no pivots for this trajectory</i>")
    color = {1:"#ffe5b4", 2:"#fff3cd", 3:"#d4edda"}
    cards = []
    for p in piv.itertuples():
        r = t[t.v_old == p.v_old].iloc[0]
        sel = "border:2px solid #e67e22;" if bp is not None and p.v_old == bp else "border:1px solid #ddd;"
        cards.append(f"""
        <div style='{sel}border-left:6px solid {color.get(int(p.confidence), "#eee")};padding:8px 12px;margin:6px 0;font-size:13px'>
          <b>v{p.v_old}→v{p.v_new}</b> &nbsp; conf <b>{p.confidence}</b> &nbsp; tk=<b>{_esc(p.tk_ids)}</b> &nbsp;
          evidence={_esc(p.evidence_type)} &nbsp; pivot={p.is_pivot} &nbsp; {fmt_score(r.score_old)}→{fmt_score(r.score_new)} ({r.score_effect})
          <p><b>expert did:</b> {_esc(r.goal_nl)}</p>
          <p><b>implicit condition:</b> {_esc(p.implicit_condition)}</p>
          <p><b>novice alternative:</b> {_esc(p.novice_alternative)}</p>
          <p>{_esc(p.note)}</p>
          <details><summary>diff</summary><pre style='font-size:11px;max-height:300px;overflow:auto'>{_esc(expert_diff(r))}</pre></details>
        </div>""")
    return HTML("<h4>Annotated pivots in this trajectory</h4>" + "".join(cards))

def refresh_traj_options(*_):
    piv = load_pivots()
    sub = TRAJ[TRAJ.comp == w_comp.value]
    counts = piv.groupby("key_id").size()
    sub = sub.assign(n_piv=sub.key_id.map(counts).fillna(0).astype(int))
    sub = sub[sub.n_piv > 0].sort_values(["n_piv","best"], ascending=[False, False])
    opts = [(f"{r.key_id} | {r.group} | {r.n_piv} pivots | {r.n_trans} trans | best={'' if pd.isna(r.best) else round(r.best,4)}", r.key_id)
            for r in sub.itertuples()]
    w_traj.options = opts or [("(no annotated trajectory in this competition)", None)]
    w_traj.value = w_traj.options[0][1]
    on_traj_change()

def on_traj_change(*_):
    if w_traj.value is None:
        w_bp.options = [("(none)", None)]; w_bp.value = None
        with out_table: clear_output()
        return
    piv = load_pivots(); piv = piv[piv.key_id == str(w_traj.value)].sort_values("v_old")
    w_bp.options = [(f"v{r.v_old}  conf {r.confidence}  {r.tk_ids}", int(r.v_old)) for r in piv.itertuples()] or [("(none)", None)]
    w_bp.value = w_bp.options[0][1]
    draw()

def draw(*_):
    if w_traj.value is None: return
    t = trajectory(w_traj.value)
    with out_table:
        clear_output(wait=True)
        display(render_traj_table(t, w_bp.value))
    with out_pivots:
        clear_output(wait=True)
        display(render_pivot_cards(t, w_bp.value))

w_comp.observe(refresh_traj_options, "value")
w_traj.observe(on_traj_change, "value")
w_bp.observe(draw, "value")
refresh_traj_options()

display(W.VBox([w_comp, w_traj, w_bp, out_pivots, out_table]))

## 7. One-click: Generate and Compare at Current Breakpoint

In [ ]:
w_go = W.Button(description="Generate and Compare at Breakpoint", button_style="primary")
w_n  = W.Dropdown(options=[1,2,3,4,5], value=CFG["N_SAMPLES"], description="samples")
w_hist = W.Dropdown(options=["diffs","full","none"], value=CFG["HISTORY_CODE"], description="snapshots")
w_reas = W.Checkbox(value=CFG["HISTORY_REASONING"], description="reasoning between snapshots")
w_hide = W.Checkbox(value=CFG["HIDE_SCORES"], description="hide scores")
w_hint = W.Textarea(value=CFG["HINT"], placeholder="Inject knowledge here (null by default), e.g., 'The public LB is computed on ...; when CV and LB diverge this much, ...'",
                    description="HINT", layout=W.Layout(width="95%", height="70px"))
out_cmp = W.Output()

def on_go(_):
    CFG["HISTORY_CODE"] = w_hist.value; CFG["HISTORY_REASONING"] = w_reas.value; CFG["HIDE_SCORES"] = w_hide.value; CFG["HINT"] = w_hint.value
    with out_cmp:
        clear_output(wait=True)
        print(f"generating at {w_traj.value} v_t={w_bp.value} ... ({w_n.value} sample(s), snapshots={w_hist.value}, reasoning={w_reas.value})")
        res = generate_at_breakpoint(w_traj.value, w_bp.value, CFG, n=w_n.value)
        if CFG["JUDGE_ENABLED"]: judge_result(res)
        save_result(res)
        clear_output(wait=True)
        display(render_comparison(res)); display(render_judgement(res))

w_go.on_click(on_go)
display(W.VBox([W.HBox([w_go, w_n, w_hist, w_reas, w_hide]), w_hint, out_cmp]))

## 8. Batch: Run all candidate breakpoints for a trajectory

First, use labels to filter candidate breakpoints (default: `magnitude ∈ {micro, minor}` and `score_effect == improving`), then generate for each. Results are written to `results.jsonl`, which can then be used in another notebook for evaluation.

In [ ]:
def candidate_breakpoints(t, sizes=("micro","minor"), effects=("improving",)):
    m = t.magnitude.isin(sizes) & t.score_effect.isin(effects)
    return list(t[m].v_old)

def run_trajectory(key_id, bps=None, cfg=CFG, show=True):
    t = trajectory(key_id)
    bps = bps or candidate_breakpoints(t)
    print(f"{key_id}: {len(bps)} breakpoints -> {bps}")
    results = []
    for bp in bps:
        res = generate_at_breakpoint(key_id, bp, cfg)
        if CFG["JUDGE_ENABLED"]:
            judge_result(res)
        save_result(res)
        results.append(res)
        if show: display(render_comparison(res))
    return results

# Example (uncomment to run):
# _ = run_trajectory(w_traj.value)

## 9. Quick Review of Saved Results

In [ ]:
def load_results(path=CFG["RESULTS_PATH"]):
    if not os.path.exists(path): return pd.DataFrame()
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    flat = []
    for r in rows:
        exp = r.get("expert") or {}
        exp_cats = set(json.loads(exp["coarse_actions"])) if isinstance(exp.get("coarse_actions"), str) else set()
        for i, o in enumerate(r["outputs"]):
            cats = set(o.get("next_action_category", []) or [])
            flat.append(dict(ts=r["ts"], comp=r["comp"], key_id=r["key_id"], bp=r["breakpoint"], sample=i,
                             snapshots=r["cfg"].get("HISTORY_CODE"), reasoning=r["cfg"].get("HISTORY_REASONING"), hint=bool(r["cfg"].get("HINT")), model=r["cfg"]["MODEL"],
                             expert_cats=",".join(sorted(exp_cats)), model_cats=",".join(sorted(cats)),
                             coarse_overlap=bool(cats & exp_cats),
                             expert_goal=exp.get("goal_nl"), model_edit=o.get("next_edit")))
    return pd.DataFrame(flat)

RES = load_results()
if len(RES):
    display(RES.tail(20))
    print("coarse-category overlap rate:", round(RES.coarse_overlap.mean(), 3), f"(n={len(RES)})")
else:
    print("results.jsonl is empty.")

results.jsonl is empty.


## Notes

- **What the model sees**: When `history=labels`, the history consists of TraceML's `diff_summary` and scores, not the raw diff. This is less information than the expert had at the time (no notebook output, no validation logs), but also more (summary written by LLM). For formal experiments, this should be stated in the setup; `labels+diffs` and `none` are two controls.
- **Expert's "answer"**: `goal_nl` and `diff_summary` are LLM annotations; the real diff is the ground truth. When comparing, prioritize the diff.
- **Coarse category overlap** is just a rough, automatic signal for a quick scan; to determine if a breakpoint contains tacit knowledge, one must read the difference between the model's `observations` and the expert's diff.
- **Score direction**: For some competitions, a lower score is better (the `competitions.json` file has a direction field); the curve should be read in reverse accordingly.